# Modul 09: Analisis Klaster (K-Means dan Klasterisasi Hirarki)
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📖 1. Analisis Klaster (K-Means dan Klasterisasi Hirarki)

Analisis Klaster (*Cluster Analysis*) adalah teknik pembelajaran tanpa pengawasan (*Unsupervised Learning*) untuk mengelompokkan objek data ke dalam klaster-klaster homogen di mana objek dalam satu klaster memiliki kemiripan tinggi (*high intra-cluster similarity*), namun sangat berbeda dengan objek pada klaster lain (*low inter-cluster similarity*):
1. **Klasterisasi Hirarki (*Hierarchical Agglomerative Clustering*)**:
   - Menggabungkan data secara bertingkat mulai dari titik individu hingga membentuk satu pohon struktur visual (**Dendrogram**).
   - Menggunakan metrik jarak (Euclidean / Manhattan) dan metode penghubung (*Ward's Linkage* atau *Complete Linkage*).
2. **K-Means Clustering**:
   - Mempartisi data ke dalam $K$ kelompok dengan mengoptimalkan posisi titik pusat (**Centroid $\mu_k$**) guna meminimalkan inersia kuadrat galat (*Within-Cluster Sum of Squares* / WCSS):
     $$	ext{WCSS} = \sum_{k=1}^K \sum_{x \in C_k} ||x - \mu_k||^2$$
3. **Validasi Jumlah Klaster Optimal**:
   - **Elbow Method**: Memilih nilai $K$ pada titik patahan siku penurunan inersia.
   - **Silhouette Score**: Mengukur pemisahan klaster dalam rentang $-1.0 \le s \le +1.0$ (skor $> 0.5$ menandakan klaster sangat terdefinisi kuat).


## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Klasterisasi K-Means & Dendrogram](images/img_09_kmeans_clustering.png)

> 🇮🇩 **Versi Bahasa Indonesia:** [Lihat Gambar Ilustrasi Versi Bahasa Indonesia (Infografis 2D)](images_id/K_Means_Cluster_Analysis_Infogra…_202608311101.jpeg)

> **Deskripsi Visual Infografis 2D:**
> 1. **Euclidean Customer Partitioning**: Partisi 3 kelompok persona nasabah (*VIP Investors*, *Business Owners*, *Student Savers*) berdasarkan jarak Euclidean ke titik pusat (*centroid*) bintang.
> 2. **Elbow Method & Silhouette Quality**: Penentuan jumlah klaster optimal pada titik siku **$K=3$** serta validasi kualitas pemisahan segmen dengan **Silhouette Score = 0.68**.



## 🔬 3. Studi Kasus & Penjelasan Langkah Komputasi

Studi kasus melakukan segmentasi perilaku belanja 200 pelanggan e-commerce (`07_customer_segmentation_clustering.csv`) berdasarkan pendapatan tahunan dan skor intensitas belanja.

**Tahapan Komputasi:**
1. Melakukan standardisasi skala fitur.
2. Membentuk Dendrogram Hirarki metode Ward.
3. Menjalankan iterasi K-Means ($K=1$ s.d. $K=8$) untuk menyusun kurva Elbow dan menghitung Silhouette Score.
4. Memberikan label persona bisnis pada setiap klaster yang terbentuk.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
# [Google Colab] Jika menjalankan di Colab, gunakan tautan direct raw GitHub berikut:
# url_07_customer_segmentation_clustering = "https://raw.githubusercontent.com/rdwnilyas-coder/statistika-komputasi-unjani/refs/heads/main/datasets/07_customer_segmentation_clustering.csv"
# df_segment = pd.read_csv(url_07_customer_segmentation_clustering)
df_segment = pd.read_csv("../datasets/07_customer_segmentation_clustering.csv")
print("Data segmentasi pelanggan dimuat:", df_segment.shape)
display(df_segment.head())


## 💻 4. Eksekusi Komputasi Python: Evaluasi & Pemodelan Klaster


In [ ]:
# 1. Standardisasi dan Penentuan K Optimal via Elbow & Silhouette
feat_cluster = ['annual_income_million', 'spending_score_1_100']
X_cluster = df_segment[feat_cluster]
X_scaled = StandardScaler().fit_transform(X_cluster)

inertias = []
sil_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

# 2. Eksekusi Model Final K=4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df_segment['Cluster'] = kmeans_final.fit_predict(X_scaled)
persona_map = {0: 'Budget Conscious', 1: 'VIP Premium Spender', 2: 'Digital Trendsetter', 3: 'Mainstream Practical'}
df_segment['Persona'] = df_segment['Cluster'].map(persona_map)

print("=== Ringkasan Karakteristik Persona Pelanggan ===")
display(df_segment.groupby('Persona')[feat_cluster].mean().round(1))


In [ ]:
# 3. Visualisasi Dendrogram, Elbow Curve, dan Partisi Klaster
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: Elbow Curve
axes[0].plot(list(K_range), inertias, 'o-', color='#1A365D', lw=2.5, markersize=8)
axes[0].axvline(4, color='#EA580C', linestyle='--', label='K Optimal = 4 (Silhouette: 0.65)')
axes[0].set_title('Evaluasi Jumlah Klaster Optimal (Elbow Method)', fontweight='bold')
axes[0].set_xlabel('Jumlah Klaster (K)')
axes[0].set_ylabel('Inersia WCSS')
axes[0].legend()

# Subplot 2: Sebaran Klaster Pelanggan
sns.scatterplot(data=df_segment, x='annual_income_million', y='spending_score_1_100', hue='Persona', 
                palette='tab10', s=80, ax=axes[1])
axes[1].set_title('Partisi Klaster Pelanggan E-Commerce (K = 4)', fontweight='bold')
axes[1].set_xlabel('Pendapatan Tahunan (Juta IDR)')
axes[1].set_ylabel('Skor Belanja (1 - 100)')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


## 📝 5. Kesimpulan Analisis & Data Storytelling

### ❓ Pertanyaan Refleksi & Konsep
* **Apa kelemahan mendasar K-Means dan bagaimana solusinya?** K-Means mengasumsikan klaster berbentuk bulat (*spherical*) dengan ukuran seragam dan sensitif terhadap inisialisasi centroid awal. Solusinya: gunakan algoritma **K-Means++** (default pada `scikit-learn`) atau **DBSCAN** untuk klaster berbentuk non-linier.

### 🔍 Temuan Utama Data (Key Findings)
* Pelanggan terbagi menjadi **4 kuadran persona yang terpisah tegas** dengan nilai Silhouette Score tinggi (**0.65**).
* Segmen *VIP Premium Spender* mencatatkan rata-rata pendapatan $>88$ Juta IDR dengan skor belanja konsisten di atas 80 poin.

### 💡 Rekomendasi & Langkah Lanjutan
* Tim pemasaran dapat menyusun *targeted loyalty campaign* spesifik untuk segmen VIP dan program diskon terarah untuk segmen *Budget Conscious*.
